# Chapter 7 — Backtesting the Entire Distribution Using PIT

教材 pp. 89–100；PDF 页序为教材页码加 11。

真实银行子组合数据未随教材提供，表7.1–7.10的通过比例不伪造。PIT 采用预测分布，不用当期实现值反向拟合预测参数。

下列未明确标为教材例题的数据均为合成验证数据，不声称复现教材中的原始市场样本。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from pit_calculator import *

均匀网格有可手算的 KS=1/(2n)、CVM=1/(12n)；pp.98–99。

In [2]:
n=100
data=pd.DataFrame({'u':(np.arange(n)+.5)/n})
out=pit_diagnostics(data,'u')
assert_allclose(out['ks'],1/(2*n))
assert_allclose(out['cvm'],1/(12*n))
assert np.isfinite(out['ad'])
print(out)

ks             0.005000
ks_p_value     1.000000
cvm            0.000833
cvm_p_value    1.000000
ad             0.011495
dtype: float64


逐期 CDF 回代检验；p.89。

In [3]:
data=pd.DataFrame({'x':norm.ppf([.1,.5,.9]),'mean':0.,'std':1.})
out=probability_integral_transform(data,'x',lambda x,d:norm.cdf(x,loc=d['mean'],scale=d['std']))
assert_allclose(out.pit,[.1,.5,.9])

## 异常的自相关、Logit/LPM及PIT独立性；pp.95–99

In [4]:
rng=np.random.default_rng(123)
x=rng.normal(size=2000)
from scipy.special import expit
y=(rng.random(2000)<expit(-.5+.8*x)).astype(int)
data=pd.DataFrame({'x':x,'hit':y})
fit=exception_logit(data,'hit',['x'])
assert_allclose(fit.coefficient,[-.5,.8],atol=.15)
assert (linear_probability_model(data,'hit',['x']).standard_error>0).all()
assert ljung_box(pd.DataFrame({'x':np.tile([0.,1.],100)}),'x',5)['p_value']<.001
data['u']=norm.cdf(.2+.4*x+rng.normal(size=len(x)))
fit=pit_independence(data,'u',['x'])
assert_allclose(fit.coefficient,[.2,.4],atol=.08)
durations=duration_test(pd.DataFrame({'hit':(rng.random(2000)<.05).astype(int)}),'hit',.05)
assert np.isfinite(durations).all()
print(durations)

shape        1.020464
intensity    0.046641
statistic    0.420090
p_value      0.810548
dtype: float64
